In [1]:
from march.pyt.mc import mc

import numpy as np
import torch
import time

# Pytorch

In [2]:
def create_voxel_grid_loop(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices.
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_x+1, res_y+1, res_z+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Create cube indices
    # For each cube at position (i, j, k), the 8 vertices follow the binary pattern:
    # (i+0, j+0, k+0), (i+1, j+0, k+0), (i+0, j+1, k+0), (i+1, j+1, k+0),
    # (i+0, j+0, k+1), (i+1, j+0, k+1), (i+0, j+1, k+1), (i+1, j+1, k+1)
    
    cube_indices = []
    for k in range(res_z):
        for j in range(res_y):
            for i in range(res_x):
                # Linear indices into the flattened grid
                stride_x = 1
                stride_y = res_x + 1
                stride_z = (res_y + 1) * (res_x + 1)
                
                v0 = k * stride_z + j * stride_y + i * stride_x
                v1 = k * stride_z + j * stride_y + (i + 1) * stride_x
                v2 = k * stride_z + (j + 1) * stride_y + i * stride_x
                v3 = k * stride_z + (j + 1) * stride_y + (i + 1) * stride_x
                v4 = (k + 1) * stride_z + j * stride_y + i * stride_x
                v5 = (k + 1) * stride_z + j * stride_y + (i + 1) * stride_x
                v6 = (k + 1) * stride_z + (j + 1) * stride_y + i * stride_x
                v7 = (k + 1) * stride_z + (j + 1) * stride_y + (i + 1) * stride_x
                
                cube_indices.append([v0, v1, v2, v3, v4, v5, v6, v7])
    
    cubes = torch.tensor(cube_indices, dtype=torch.long, device=device)
    
    return grids, cubes

In [3]:
def create_voxel_grid_torch(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices (vectorized).
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_z+1, res_y+1, res_x+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Vectorized cube indices generation
    stride_x = 1
    stride_y = res_x + 1
    stride_z = (res_y + 1) * (res_x + 1)
    
    # Create all cube positions via meshgrid
    i_idx = torch.arange(res_x, device=device)
    j_idx = torch.arange(res_y, device=device)
    k_idx = torch.arange(res_z, device=device)

    k_grid, j_grid, i_grid = torch.meshgrid(k_idx, j_idx, i_idx, indexing='ij')
    
    # Compute base index for each cube
    base = k_grid * stride_z + j_grid * stride_y + i_grid * stride_x
    base = base.flatten().unsqueeze(1)  # shape (num_cubes, 1)
    
    # Define vertex offsets within a cube
    offsets = torch.tensor([
        [0, 0, 0],  # v0
        [1, 0, 0],  # v1
        [0, 1, 0],  # v2
        [1, 1, 0],  # v3
        [0, 0, 1],  # v4
        [1, 0, 1],  # v5
        [0, 1, 1],  # v6
        [1, 1, 1],  # v7
    ], dtype=torch.long, device=device)
    
    # Convert offsets to linear indices
    vertex_offsets = offsets[:, 0] * stride_x + offsets[:, 1] * stride_y + offsets[:, 2] * stride_z
    
    # Broadcast and add: shape (num_cubes, 8)
    cubes = base + vertex_offsets.unsqueeze(0)
    cubes = cubes.long()
    
    return grids, cubes

In [4]:
def create_voxel_grid_torch_fused(res_x, res_y, res_z, bounds, dtype=torch.float32, device='cpu'):
    """
    Create a voxel grid with vertices and cube indices (vectorized).
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: torch.float32)
    
    Returns:
        grids: Tensor of shape (num_vertices, 3) containing vertex coordinates
        cubes: Tensor of shape (num_cubes, 8) containing vertex indices for each cube
    """
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = torch.linspace(x_min, x_max, res_x + 1, dtype=dtype, device=device)
    y_coords = torch.linspace(y_min, y_max, res_y + 1, dtype=dtype, device=device)
    z_coords = torch.linspace(z_min, z_max, res_z + 1, dtype=dtype, device=device)
    
    # Create meshgrid: shape (res_z+1, res_y+1, res_x+1)
    zz, yy, xx = torch.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = torch.stack([xx.flatten(), yy.flatten(), zz.flatten()], dim=1)
    
    # Vectorized cube indices generation
    stride_x = 1
    stride_y = res_x + 1
    stride_z = (res_y + 1) * (res_x + 1)
    
    # Create all cube positions via meshgrid
    i_idx = torch.arange(res_x, device=device)
    j_idx = torch.arange(res_y, device=device)
    k_idx = torch.arange(res_z, device=device)

    k_grid, j_grid, i_grid = torch.meshgrid(k_idx, j_idx, i_idx, indexing='ij')
    
    # Define vertex offsets within a cube
    offsets = torch.tensor([
        [0, 0, 0],  # v0
        [1, 0, 0],  # v1
        [0, 1, 0],  # v2
        [1, 1, 0],  # v3
        [0, 0, 1],  # v4
        [1, 0, 1],  # v5
        [0, 1, 1],  # v6
        [1, 1, 1],  # v7
    ], dtype=torch.long, device=device)
    
    # Convert offsets to linear indices
    vertex_offsets = offsets[:, 0] * stride_x + offsets[:, 1] * stride_y + offsets[:, 2] * stride_z
    
    # Broadcast and add: shape (num_cubes, 8)
    cubes = (k_grid * stride_z + j_grid * stride_y + i_grid * stride_x).flatten().unsqueeze(1) + vertex_offsets.unsqueeze(0)
    cubes = cubes.long()
    
    return grids, cubes

In [5]:
res_x, res_y, res_z = 2, 1, 2

bounds = ((0, 2), (0, 1), (-1, 1))

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, bounds)

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

print("Grid Vertices:\n", grids_loop, "\nShape:", grids_loop.shape)
print("Cubes:\n", cubes_loop, "\nShape:", cubes_loop.shape)

Time taken: 0.0006 seconds
Grid Vertices:
 tensor([[ 0.,  0., -1.],
        [ 1.,  0., -1.],
        [ 2.,  0., -1.],
        [ 0.,  1., -1.],
        [ 1.,  1., -1.],
        [ 2.,  1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  0.,  0.],
        [ 2.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 1.,  1.,  0.],
        [ 2.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  1.],
        [ 2.,  0.,  1.],
        [ 0.,  1.,  1.],
        [ 1.,  1.,  1.],
        [ 2.,  1.,  1.]]) 
Shape: torch.Size([18, 3])
Cubes:
 tensor([[ 0,  1,  3,  4,  6,  7,  9, 10],
        [ 1,  2,  4,  5,  7,  8, 10, 11],
        [ 6,  7,  9, 10, 12, 13, 15, 16],
        [ 7,  8, 10, 11, 13, 14, 16, 17]]) 
Shape: torch.Size([4, 8])


In [6]:
time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, bounds)

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

print("Grid Vertices:\n", grids_torch, "\nShape:", grids_torch.shape)
print("Cubes:\n", cubes_torch, "\nShape:", cubes_torch.shape)

Time taken: 0.0004 seconds
Grid Vertices:
 tensor([[ 0.,  0., -1.],
        [ 1.,  0., -1.],
        [ 2.,  0., -1.],
        [ 0.,  1., -1.],
        [ 1.,  1., -1.],
        [ 2.,  1., -1.],
        [ 0.,  0.,  0.],
        [ 1.,  0.,  0.],
        [ 2.,  0.,  0.],
        [ 0.,  1.,  0.],
        [ 1.,  1.,  0.],
        [ 2.,  1.,  0.],
        [ 0.,  0.,  1.],
        [ 1.,  0.,  1.],
        [ 2.,  0.,  1.],
        [ 0.,  1.,  1.],
        [ 1.,  1.,  1.],
        [ 2.,  1.,  1.]]) 
Shape: torch.Size([18, 3])
Cubes:
 tensor([[ 0,  1,  3,  4,  6,  7,  9, 10],
        [ 1,  2,  4,  5,  7,  8, 10, 11],
        [ 6,  7,  9, 10, 12, 13, 15, 16],
        [ 7,  8, 10, 11, 13, 14, 16, 17]]) 
Shape: torch.Size([4, 8])


In [7]:
print(torch.allclose(grids_loop, grids_torch))  # Should be True
print(torch.equal(cubes_loop, cubes_torch))  # Should be True

True
True


In [8]:
res_x, res_y, res_z = 128, 128, 128

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cpu')

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cpu')

time_end = time.time()

print(f"Time taken: {time_end - time_start:.4f} seconds")

torch.cuda.empty_cache()

Time taken: 2.4737 seconds
Time taken: 0.0129 seconds


In [9]:
res_x, res_y, res_z = 128, 128, 128

time_start = time.time()
grids_loop, cubes_loop = create_voxel_grid_loop(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cuda')

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

time_start = time.time()
grids_torch, cubes_torch = create_voxel_grid_torch(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), device='cuda')

time_end = time.time()

print(f"Time taken: {time_end - time_start:.4f} seconds")

torch.cuda.empty_cache()

Time taken: 2.3846 seconds
Time taken: 0.0043 seconds


# Numpy

In [10]:
def create_voxel_grid_numpy(res_x, res_y, res_z, bounds, dtype=np.float32):
    """
    Create a voxel grid with vertices and cube indices (vectorized).
    
    Args:
        res_x, res_y, res_z: Resolution (number of voxels) in each dimension
        bounds: Tuple of ((x_min, x_max), (y_min, y_max), (z_min, z_max))
        dtype: Data type for vertex coordinates (default: np.float32)
    
    Returns:
        grids: Numpy array of shape (num_vertices, 3) containing vertex coordinates
        cubes: Numpy array of shape (num_cubes, 8) containing vertex indices for each cube
    """
    
    (x_min, x_max), (y_min, y_max), (z_min, z_max) = bounds
    
    # Create 1D coordinates for each dimension
    x_coords = np.linspace(x_min, x_max, res_x + 1, dtype=dtype)
    y_coords = np.linspace(y_min, y_max, res_y + 1, dtype=dtype)
    z_coords = np.linspace(z_min, z_max, res_z + 1, dtype=dtype)
    
    # Create meshgrid: shape (res_z+1, res_y+1, res_x+1)
    zz, yy, xx = np.meshgrid(z_coords, y_coords, x_coords, indexing='ij')
    
    # Flatten to get all vertices: shape (num_vertices, 3)
    grids = np.stack([xx.flatten(), yy.flatten(), zz.flatten()], axis=1)
    
    # Vectorized cube indices generation
    stride_x = 1
    stride_y = res_x + 1
    stride_z = (res_y + 1) * (res_x + 1)
    
    # Create all cube positions via meshgrid
    i_idx = np.arange(res_x)
    j_idx = np.arange(res_y)
    k_idx = np.arange(res_z)
    
    k_grid, j_grid, i_grid = np.meshgrid(k_idx, j_idx, i_idx, indexing='ij')
    
    # Compute base index for each cube
    base = k_grid * stride_z + j_grid * stride_y + i_grid * stride_x
    base = np.expand_dims(base.flatten(), axis=1)  # shape (num_cubes, 1)
    
    # Define vertex offsets within a cube
    offsets = np.array([
        [0, 0, 0],  # v0
        [1, 0, 0],  # v1
        [0, 1, 0],  # v2
        [1, 1, 0],  # v3
        [0, 0, 1],  # v4
        [1, 0, 1],  # v5
        [0, 1, 1],  # v6
        [1, 1, 1],  # v7
    ], dtype=np.int64)
    
    # Convert offsets to linear indices
    vertex_offsets = offsets[:, 0] * stride_x + offsets[:, 1] * stride_y + offsets[:, 2] * stride_z
    
    # Broadcast and add: shape (num_cubes, 8)
    cubes = base + np.expand_dims(vertex_offsets, axis=0)
    cubes = cubes.astype(np.int64)
    
    return grids, cubes

In [11]:
res_x, res_y, res_z = 128, 128, 128

time_start = time.time()
grids_np, cubes_np = create_voxel_grid_numpy(res_x, res_y, res_z, ((-1, 1), (-1, 1), (-1, 1)), dtype=np.float32)

time_end = time.time()
print(f"Time taken: {time_end - time_start:.4f} seconds")

Time taken: 0.0682 seconds
